In [ ]:
import pandas as pd
import seaborn as sns
import tifffile
from matplotlib import pyplot as plt
import numpy as np
import math
import os
import torch
from tqdm import tqdm

# Load H&E Image

In [ ]:
dataset = "Xe_hsa_skin_h"
data_dir = f"PATH/TO/{dataset}/"
image = tifffile.imread(f"{data_dir}he-raw.tiff")

In [ ]:
print(image.shape)
plt.imshow(image, cmap='binary')
plt.title('H&E image', size=16)
plt.axis('scaled')
plt.show()

# Load Transcript Coordinates

In [ ]:
transcripts_all = pd.read_csv(
    f"{data_dir}transcripts_use.tsv",
    sep='\t',
    header=0
)

In [ ]:
print("Number of gene:\t\t" + str(len(np.unique(transcripts_all['feature_name']))))
print("Number of transcript:\t" + str(len(transcripts_all)))

# Simulate different drop-out ratio

In [ ]:
def trans_sample(group, fraction, seed):
    total_size = len(group)
    sample_size = int(np.floor(total_size * fraction))
    sampled_group = group.sample(n=sample_size, random_state=seed)
    return sampled_group

In [ ]:
drop_out_ratio_all = [0.1, 0.2, 0.3, 0.4, 0.5]
seeds = [123, 124, 125]
for dor in drop_out_ratio_all:
    for i in range(3):
        seed = seeds[i]
        dor_name = str(dor)
        transcripts_dor = pd.concat([trans_sample(group, 1-dor, seed) for _, group in transcripts_all.groupby('feature_name', sort=True)], axis=0)
        transcripts_dor = transcripts_dor.reset_index(drop=True)
        if not os.path.exists(f"{data_dir}drop_out/transcripts/"):
            os.makedirs(f"{data_dir}drop_out/transcripts/")
        transcripts_dor.to_csv(f"{data_dir}drop_out/transcripts/transcripts_dor{dor_name}_rep{i+1}.tsv", sep='\t', index=False)

# Load spot location

In [ ]:
spots = pd.read_csv(
    f"{data_dir}locs-raw.tsv",
    sep='\t',
    header=0,
    index_col=0
)
spots

In [ ]:
drop_out_ratio_all = [0.1, 0.2, 0.3, 0.4, 0.5]
radius_use = 27.5 / 1.7
device = 'cuda' if torch.cuda.is_available() else 'cpu'
spot_xy = torch.tensor(spots[['x', 'y']].values, dtype=torch.float32, device=device)
for dor in drop_out_ratio_all:
    dor_name = str(dor)
    for repeat in range(1,4):
        transcripts_use = pd.read_csv(
            f"{data_dir}drop_out/transcripts/transcripts_dor{dor_name}_rep{repeat}.tsv",
            sep='\t',
            header=0
        )
        print("Working on drop-out ratio " + dor_name + " rep " + str(repeat))
        transcript_xy = torch.tensor(transcripts_use[['x_location', 'y_location']].values, dtype=torch.float32, device=device)
        transcript_gene = transcripts_use['feature_name'].astype('category')
        gene_ids = torch.tensor(transcript_gene.cat.codes.values, device=device)
        all_gene_names = transcript_gene.cat.categories.tolist()
        cnts = torch.zeros((spot_xy.shape[0], len(all_gene_names)), dtype=torch.int32, device=device)
        for i in tqdm(range(spot_xy.shape[0])):
            center = spot_xy[i]  # shape [2]
            distances = torch.norm(transcript_xy - center, dim=1)  # [num_transcripts]
            mask = distances <= radius_use
            selected_genes = gene_ids[mask].long()  
            if selected_genes.numel() > 0:
                cnts[i].scatter_add_(0, selected_genes, torch.ones_like(selected_genes, dtype=torch.int32))
        cnts_df = pd.DataFrame(cnts.cpu().numpy(), index=spots.index, columns=all_gene_names)
        if not os.path.exists(f"{data_dir}drop_out/cnts/"):
            os.makedirs(f"{data_dir}drop_out/cnts/")
        cnts_df.index.name = 'spot'
        cnts_df.to_csv(f"{data_dir}drop_out/cnts/cnts{dor_name}_rep{repeat}.tsv", sep='\t', index=True)